# 02 — Frozen baseline intermediate smoke training
Run notebook 00 first and keep the same Colab GPU kernel. Notebook 00 downloads/caches the official combined 4,040-image training set and generates its manifest. This notebook runs the protocol's intermediate 256-image/five-epoch smoke job for both frozen backbones. No test set is used.

In [1]:
PROJECT_DIR = '/content/cod-ssl'
RUNS_ROOT = '/content/drive/MyDrive/cod-ssl/runs'
SAMPLE_IMAGE = '/content/cod_ssl_sample_image.png'
TRAIN_MANIFEST = f'{PROJECT_DIR}/manifests/train_all.csv'
LIMIT_TRAIN = 256
EPOCHS = 5

In [2]:
# Restore environment variables if this notebook was attached to a fresh kernel.
import os
os.environ.setdefault('DINOV3_REPO_DIR','/content/third_party/dinov3')
os.environ.setdefault('DINOV3_WEIGHTS','/content/drive/MyDrive/cod-ssl/checkpoints/dinov3_vitb16.pth')
os.environ.setdefault('VJEPA2_REPO_DIR','/content/third_party/vjepa2')
os.environ.setdefault('VJEPA21_WEIGHTS','/content/drive/MyDrive/cod-ssl/checkpoints/vjepa2_1_vitb_dist_vitG_384.pt')
from pathlib import Path
import torch
required=[PROJECT_DIR,TRAIN_MANIFEST,os.environ['DINOV3_WEIGHTS'],os.environ['VJEPA21_WEIGHTS']]
missing=[path for path in required if not Path(path).exists()]
if missing: raise FileNotFoundError('Run all cells in notebook 00 first. Missing:\n'+'\n'.join(missing))
if not torch.cuda.is_available(): raise RuntimeError('Connect to a GPU-backed Colab kernel.')
print('GPU:',torch.cuda.get_device_name(0))

GPU: NVIDIA A100-SXM4-40GB


In [3]:
# Pull the latest project and reinstall it in the current kernel.
import subprocess, sys
subprocess.run(['git','-C',PROJECT_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{PROJECT_DIR}[dev,notebooks]'],check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-e', '/content/cod-ssl[dev,notebooks]'], returncode=0)

In [4]:
# Verify the bootstrap-created standard training manifest before smoke training.
import pandas as pd
train_all=pd.read_csv(TRAIN_MANIFEST)
counts=train_all.groupby('source').size().to_dict()
if counts!={'camo':1000,'cod10k':3040}: raise ValueError(f'Unexpected training counts: {counts}')
missing_pairs=[path for column in ('image_path','mask_path') for path in train_all[column] if not Path(path).is_file()]
if missing_pairs: raise FileNotFoundError(f'Manifest contains missing files, first: {missing_pairs[0]}')
print(counts); print('Total:',len(train_all))

{'camo': 1000, 'cod10k': 3040}
Total: 4040


In [5]:
# Ensure the automatically downloaded smoke-test image exists for checkpoint reload verification.
from urllib.request import urlretrieve
if not Path(SAMPLE_IMAGE).is_file():
    urlretrieve('https://raw.githubusercontent.com/DengPingFan/SINet/master/Images/CamouflagedTask.png',SAMPLE_IMAGE)
print('Sample image:',SAMPLE_IMAGE)

Sample image: /content/cod_ssl_sample_image.png


In [6]:
# Run frozen DINOv3 + common decoder with the configured intermediate smoke settings.
dino_before=set(Path(RUNS_ROOT).glob('*_dinov3_vitb16_seed42'))
result=subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/train.py','--config',f'{PROJECT_DIR}/configs/frozen_dinov3_vitb16.yaml','--runs-root',RUNS_ROOT,'--limit-train',str(LIMIT_TRAIN),'--epochs',str(EPOCHS)],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError(f'DINOv3 smoke training failed: {result.returncode}')
dino_new=set(Path(RUNS_ROOT).glob('*_dinov3_vitb16_seed42'))-dino_before
if len(dino_new)!=1: raise RuntimeError(f'Could not identify DINOv3 run: {dino_new}')
DINO_RUN=str(dino_new.pop()); print('DINO_RUN=',DINO_RUN)

2026-08-31 17:10:24.154190: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-31 17:10:24.224554: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Downloading: "file:///content/drive/MyDrive/cod-ssl/checkpoints/dinov3_vitb16.pth" to /root/.cache/torch/hub/checkpoints/dinov3_vitb16.pth

  0%|          | 0.00/327M [00:00<?, ?B/s]
  0%|          | 128k/327M [00:02<1:39:09, 57.6kB/s]
  1%|          | 4.00M/327M [00:02<02:23, 2.36MB/s] 
  2%|▏         | 8.00M/327M [00:02<01:15, 4.45MB/s]
  6%|▌         | 20.4M/327M [00:0

In [7]:
# Reload the DINOv3 checkpoint and verify finite 384×384 logits and the freeze invariant.
result=subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/verify_checkpoint.py','--run',DINO_RUN,'--image',SAMPLE_IMAGE],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError('DINOv3 checkpoint reload failed.')

{
  "checkpoint": "/content/drive/MyDrive/cod-ssl/runs/20260831T171027Z_dinov3_vitb16_seed42/checkpoints/last.pt",
  "checkpoint_epoch": 5,
  "global_step": 80,
  "logit_shape": [
    1,
    1,
    384,
    384
  ],
  "logits_finite": true,
  "backbone_trainable_parameters": 0,
  "backbone_parameters_with_gradients": 0
}



In [ ]:
# Release process-local allocator caches before V-JEPA.
import gc
gc.collect(); torch.cuda.empty_cache()

In [9]:
# Run frozen V-JEPA 2.1 + the identical common decoder and smoke settings.
vjepa_before=set(Path(RUNS_ROOT).glob('*_vjepa21_vitb16_seed42'))
result=subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/train.py','--config',f'{PROJECT_DIR}/configs/frozen_vjepa21_vitb16.yaml','--runs-root',RUNS_ROOT,'--limit-train',str(LIMIT_TRAIN),'--epochs',str(EPOCHS)],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError(f'V-JEPA smoke training failed: {result.returncode}')
vjepa_new=set(Path(RUNS_ROOT).glob('*_vjepa21_vitb16_seed42'))-vjepa_before
if len(vjepa_new)!=1: raise RuntimeError(f'Could not identify V-JEPA run: {vjepa_new}')
VJEPA_RUN=str(vjepa_new.pop()); print('VJEPA_RUN=',VJEPA_RUN)

2026-08-31 17:16:01.456828: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-31 17:16:01.526995: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Run directory: /content/drive/MyDrive/cod-ssl/runs/20260831T171604Z_vjepa21_vitb16_seed42
AMP: enab

In [10]:
# Reload the V-JEPA checkpoint and verify its output and freeze invariant.
result=subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/verify_checkpoint.py','--run',VJEPA_RUN,'--image',SAMPLE_IMAGE],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError('V-JEPA checkpoint reload failed.')

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/lib/python3.13/contextlib.py:109: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
{
  "checkpoint": "/content/drive/MyDrive/cod-ssl/runs/20260831T171604Z_vjepa21_vitb16_seed42/checkpoints/last.pt",
  "checkpoint_epoch": 5,
  "global_step": 80,
  "logit_shape": [
    1,
    1,
    384,
    384
  ],
  "logits_finite": true,
  "backbone_trainable_parameters": 0,
  "backbone_parameters_with_gradients": 0
}



In [11]:
# Summarize smoke losses and required artifacts.
import numpy as np
for name,run in [('DINOv3',DINO_RUN),('V-JEPA 2.1',VJEPA_RUN)]:
    log=pd.read_csv(Path(run)/'training_log.csv')
    if len(log)!=EPOCHS: raise RuntimeError(f'{name} logged {len(log)} epochs; expected {EPOCHS}')
    if not np.isfinite(log.loss).all(): raise RuntimeError(f'{name} produced non-finite loss')
    if log.loss.iloc[-1]>=log.loss.iloc[0]: raise RuntimeError(f'{name} loss did not decrease across the smoke run')
    required=[Path(run)/'checkpoints/last.pt',Path(run)/'samples/training_sample.png',Path(run)/'config.yaml',Path(run)/'environment.txt',Path(run)/'upstream_versions.json']
    missing=[str(path) for path in required if not path.is_file()]
    if missing: raise FileNotFoundError(f'{name} missing artifacts: {missing}')
    print(f'\n{name}: {run}'); print(log[['epoch','loss','learning_rate','wall_time_seconds']].to_string(index=False))
    print('Prediction:',Path(run)/'samples/training_sample.png')


DINOv3: /content/drive/MyDrive/cod-ssl/runs/20260831T171027Z_dinov3_vitb16_seed42
 epoch     loss  learning_rate  wall_time_seconds
   1.0 1.406803       0.000905         222.384558
   2.0 1.225040       0.000655           3.036682
   3.0 1.153117       0.000345           3.085150
   4.0 1.118984       0.000095           2.996687
   5.0 1.104994       0.000000           3.068376
Prediction: /content/drive/MyDrive/cod-ssl/runs/20260831T171027Z_dinov3_vitb16_seed42/samples/training_sample.png

V-JEPA 2.1: /content/drive/MyDrive/cod-ssl/runs/20260831T171604Z_vjepa21_vitb16_seed42
 epoch     loss  learning_rate  wall_time_seconds
   1.0 1.461545       0.000905           5.282356
   2.0 1.278561       0.000655           4.149507
   3.0 1.201479       0.000345           4.201837
   4.0 1.161773       0.000095           4.055594
   5.0 1.146437       0.000000           4.211826
Prediction: /content/drive/MyDrive/cod-ssl/runs/20260831T171604Z_vjepa21_vitb16_seed42/samples/training_sample.png


In [ ]:
# Export paired qualitative panels before authorizing the full Phase-1 runs.
SMOKE_VIS_DIR = Path(RUNS_ROOT) / 'smoke_visual_comparison'
visualization_script = Path(PROJECT_DIR) / 'scripts/visualize_smoke_comparison.py'
if not visualization_script.is_file():
    raise FileNotFoundError(
        f'{visualization_script} is missing from the Colab checkout. Commit/push the latest '
        'local changes, then rerun the notebook repository-sync cell before this cell.'
    )
result = subprocess.run([
    sys.executable, str(visualization_script),
    '--manifest', TRAIN_MANIFEST, '--dino-run', DINO_RUN, '--vjepa-run', VJEPA_RUN,
    '--output', str(SMOKE_VIS_DIR), '--count', '6', '--training-subset', str(LIMIT_TRAIN),
], cwd=PROJECT_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode:
    raise RuntimeError(f'Smoke visualization failed with exit code {result.returncode}.')
if hasattr(os, 'sync'): os.sync()
print('Smoke visual comparison:', SMOKE_VIS_DIR)

In [ ]:
# Display all six original/GT/prediction comparisons inline.
from IPython.display import display
from PIL import Image
scores = pd.read_csv(SMOKE_VIS_DIR / 'smoke_visual_scores.csv')
display(scores[['id', 'dino_dice', 'vjepa_dice']])
for panel in sorted(SMOKE_VIS_DIR.glob('*.png')):
    print(panel.name)
    display(Image.open(panel))

Milestone H passes when both jobs have finite losses, saved sample predictions, reloadable checkpoints, `[1,1,384,384]` finite logits, and zero trainable/gradient-bearing backbone parameters. Full experiments remain blocked until the paired smoke overlays above are visually inspected.